# Deliverable 3 — Fixed-accuracy LBM convergence

Runs a factorial Re × grid × Mach sweep rather than changing Re and grid together.

This notebook is an executable evidence artifact. Its default configuration is
deliberately small enough for a clean local rerun; scale-up parameters are
listed separately and are not represented as measured results.

In [1]:
from pathlib import Path
import sys

repo_root = Path.cwd().parent if Path.cwd().name == "deliverables" else Path.cwd()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
output_dir = repo_root / "results" / "deliverables"
output_dir.mkdir(parents=True, exist_ok=True)

## Measured default sweep

The sweep uses a short physical time to keep this artifact rerunnable. It
separates Reynolds number, grid resolution, and Mach number and performs two
wall-time repetitions. Final-submission runs should use \(t=1\), at least five
repetitions, and the larger configuration shown below.

In [2]:
import json
import numpy as np
import pandas as pd
from quantum_aero.classical import LBMConfig, run_lbm

rows = []
for reynolds in (10, 100, 400, 1000):
    for n in (16, 32, 64):
        for mach in (0.1, 0.05, 0.025):
            trials = [run_lbm(LBMConfig(n=n, reynolds=reynolds, t_end=0.25, mach=mach, snapshots=2)) for _ in range(2)]
            final = trials[-1]["records"][-1]
            runtimes = [trial["runtime_seconds"] for trial in trials]
            rows.append({
                "reynolds": reynolds, "n": n, "mach": mach, "t_end": 0.25,
                "steps": trials[-1]["steps"], "tau": trials[-1]["tau"],
                "runtime_median_seconds": float(np.median(runtimes)),
                "runtime_min_seconds": float(np.min(runtimes)),
                "runtime_max_seconds": float(np.max(runtimes)),
                "relative_l2": final["relative_l2"],
                "vortex_relative_l2": final["vortex_relative_l2"],
                "kinetic_energy_error": abs(final["kinetic_energy"]-final["exact_kinetic_energy"]),
                "mass_relative_drift": final["mass_relative_drift"],
                "divergence_l2": final["divergence_l2"],
            })
df = pd.DataFrame(rows)
df.to_csv(output_dir / "03_fixed_accuracy_convergence.csv", index=False)
df.head(12)

,reynolds,n,mach,t_end,steps,tau,runtime_median_seconds,runtime_min_seconds,runtime_max_seconds,relative_l2,vortex_relative_l2,kinetic_energy_error,mass_relative_drift,divergence_l2
0,10,16,0.100,0.25,25,0.519454,0.032806,0.032697,0.032915,0.017741,0.031788,0.012780,2.775558e-15,0.016272
1,10,16,0.050,0.25,50,0.509727,0.066174,0.058030,0.074319,0.013961,0.025015,0.011030,5.773160e-15,0.002288
2,10,16,0.025,0.25,99,0.504913,0.128352,0.125645,0.131060,0.012671,0.022703,0.009948,1.099121e-14,0.000515
3,10,32,0.100,0.25,50,0.538907,0.113888,0.103536,0.124241,0.008587,0.015385,0.002883,5.329071e-15,0.018260
4,10,32,0.050,0.25,99,0.519650,0.195668,0.187175,0.204161,0.003943,0.007064,0.002934,1.065814e-14,0.003446
5,10,32,0.025,0.25,198,0.509825,0.376790,0.373423,0.380157,0.003853,0.006903,0.002896,2.164935e-14,0.003140
6,10,64,0.100,0.25,99,0.578601,0.501419,0.478887,0.523952,0.008231,0.014747,0.000718,9.547918e-15,0.019580
7,10,64,0.050,0.25,198,0.539300,1.176737,1.034492,1.318983,0.002010,0.003602,0.000762,2.042810e-14,0.004245
8,10,64,0.025,0.25,395,0.519700,2.074706,2.050179,2.099233,0.001678,0.003007,0.000724,4.241052e-14,0.003389
9,100,16,0.100,0.25,25,0.501945,0.053071,0.047343,0.058798,0.025591,0.044474,0.020241,2.886580e-15,0.018585


In [3]:
tolerances = (1e-2, 5e-3)
best = []
for tolerance in tolerances:
    for reynolds in sorted(df.reynolds.unique()):
        eligible = df[(df.reynolds == reynolds) & (df.relative_l2 <= tolerance)]
        if len(eligible):
            row = eligible.sort_values("runtime_median_seconds").iloc[0]
            best.append({"tolerance": tolerance, "reynolds": reynolds, "n": int(row.n), "mach": row.mach,
                         "runtime_seconds": row.runtime_median_seconds, "relative_l2": row.relative_l2})
        else:
            best.append({"tolerance": tolerance, "reynolds": reynolds, "n": None, "mach": None,
                         "runtime_seconds": None, "relative_l2": None})
best_df = pd.DataFrame(best)
best_df.to_csv(output_dir / "03_fixed_accuracy_frontier.csv", index=False)
assert len(df) == 36
print("PASS: 36 independent configurations completed.")
best_df

PASS: 36 independent configurations completed.


,tolerance,reynolds,n,mach,runtime_seconds,relative_l2
0,0.010,10,32,0.100,0.113888,0.008587
1,0.010,100,32,0.100,0.108174,0.008685
2,0.010,400,16,0.025,0.118880,0.007113
3,0.010,1000,32,0.100,0.097259,0.008469
4,0.005,10,32,0.050,0.195668,0.003943
5,0.005,100,32,0.050,0.222403,0.004422
6,0.005,400,32,0.050,0.193400,0.004697
7,0.005,1000,32,0.050,0.207524,0.004774


## Final scale-up

Use Re = 10, 100, 400, 1000, 2000, 5000; \(N=32\)–2048; Ma =
0.1–0.0125; \(t=1\); at least five repetitions; and report observed
convergence orders plus the least-cost configuration at each declared error.